In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('PS_20174392719_1491204439457_log.csv')

In [3]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [4]:
df.shape

(6362620, 11)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


In [6]:
df.isna().sum()

,0
step,0
type,0
amount,0
nameOrig,0
oldbalanceOrg,0
newbalanceOrig,0
nameDest,0
oldbalanceDest,0
newbalanceDest,0
isFraud,0


In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03,2.514687e-06
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02,1.585775e-03
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,0.000000e+00
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+00


In [9]:
print(df['nameOrig'].nunique())
print(df['nameDest'].nunique())
print("Fraud rate:", df['isFraud'].mean())
print("Fraud percentage:", df['isFraud'].mean() * 100)

6353307
2722362
Fraud rate: 0.001290820448180152
Fraud percentage: 0.12908204481801522


In [10]:
df['isFraud'].value_counts()

,count
isFraud,
0,6354407
1,8213


In [11]:
orig_repeat = df.groupby('nameOrig').size()
print(orig_repeat.value_counts())
print("Maksimum təkrar sayı:", orig_repeat.max())

1    6344009
2       9283
3         15
Name: count, dtype: int64
Maksimum təkrar sayı: 3


In [12]:
df['balance_mismatch'] = ((df['oldbalanceOrg'] - df['amount']).round(2)!= df['newbalanceOrig'].round(2)).astype(int)
print(df['balance_mismatch'].value_counts())

balance_mismatch
1    5125552
0    1237068
Name: count, dtype: int64


In [13]:
fraud_customers = df.loc[df['isFraud'] == 1, 'nameOrig'].unique()
clean_customers = (df.loc[~df['nameOrig'].isin(fraud_customers), 'nameOrig'].drop_duplicates().sample(3000, random_state=42))
selected_customers = set(fraud_customers) | set(clean_customers)

subset = df[df['nameOrig'].isin(selected_customers)].copy()

In [14]:
print("Original shape:", df.shape)
print("Subset shape:", subset.shape)

print("Original fraud rate:", df['isFraud'].mean())
print("Subset fraud rate:", subset['isFraud'].mean())

Original shape: (6362620, 12)
Subset shape: (11247, 12)
Original fraud rate: 0.001290820448180152
Subset fraud rate: 0.730239174891082


## 5. Entity və event timestamp

In [15]:
base_date = pd.Timestamp("2024-01-01")
subset['event_timestamp'] = (base_date + pd.to_timedelta(subset['step'], unit='h'))
subset = subset.sort_values(['nameOrig', 'event_timestamp']).reset_index(drop=True)

subset[['nameOrig', 'step', 'event_timestamp']].head()

,nameOrig,step,event_timestamp
0,C1000036340,655,2024-01-28 07:00:00
1,C1000086512,95,2024-01-04 23:00:00
2,C1000331499,262,2024-01-11 22:00:00
3,C1000437286,284,2024-01-12 20:00:00
4,C1000484178,504,2024-01-22 00:00:00


In [16]:
step_agg = (subset.groupby(['nameOrig', 'step']).agg(step_txn_count=('amount', 'size'),step_amount_sum=('amount', 'sum'),step_last_amount=('amount', 'last'),).reset_index().sort_values(['nameOrig', 'step']))

step_agg.head()

,nameOrig,step,step_txn_count,step_amount_sum,step_last_amount
0,C1000036340,655,1,253648.68,253648.68
1,C1000086512,95,1,33676.59,33676.59
2,C1000331499,262,1,2016790.84,2016790.84
3,C1000437286,284,1,181182.79,181182.79
4,C1000484178,504,1,3018810.85,3018810.85


In [18]:
g_step = step_agg.groupby('nameOrig')

step_agg['cum_count_incl_current']  = g_step['step_txn_count'].cumsum()
step_agg['cum_amount_incl_current'] = g_step['step_amount_sum'].cumsum()

step_agg['txn_count_so_far']  = step_agg['cum_count_incl_current']  - step_agg['step_txn_count']
step_agg['total_sent_so_far'] = step_agg['cum_amount_incl_current'] - step_agg['step_amount_sum']

step_agg['avg_amount_so_far'] = np.where(step_agg['txn_count_so_far'] > 0,step_agg['total_sent_so_far'] / step_agg['txn_count_so_far'],0.0)

In [19]:
step_agg['last_txn_amount'] = g_step['step_last_amount'].shift(1).fillna(0)
step_agg['prev_step'] = g_step['step'].shift(1)

step_agg['hours_since_last_txn'] = (step_agg['step'] - step_agg['prev_step']).fillna(0)

In [20]:
feature_col_step = ['nameOrig', 'step', 'txn_count_so_far', 'total_sent_so_far', 'avg_amount_so_far', 'last_txn_amount', 'hours_since_last_txn']
subset = subset.merge(step_agg[feature_col_step], on=['nameOrig', 'step'], how='left')
subset.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,balance_mismatch,event_timestamp,txn_count_so_far,total_sent_so_far,avg_amount_so_far,last_txn_amount,hours_since_last_txn
0,655,TRANSFER,253648.68,C1000036340,253648.68,0.0,C1958275811,0.00,0.00,1,0,0,2024-01-28 07:00:00,0,0.0,0.0,0.0,0.0
1,95,CASH_OUT,33676.59,C1000086512,33676.59,0.0,C1759363094,0.00,33676.59,1,0,0,2024-01-04 23:00:00,0,0.0,0.0,0.0,0.0
2,262,TRANSFER,2016790.84,C1000331499,2016790.84,0.0,C1778895918,0.00,0.00,1,0,0,2024-01-11 22:00:00,0,0.0,0.0,0.0,0.0
3,284,CASH_OUT,181182.79,C1000437286,9601.00,0.0,C1827183939,150236.44,331419.24,0,0,1,2024-01-12 20:00:00,0,0.0,0.0,0.0,0.0
4,504,CASH_OUT,3018810.85,C1000484178,3018810.85,0.0,C895750711,65126.98,3083937.83,1,0,0,2024-01-22 00:00:00,0,0.0,0.0,0.0,0.0
